# CS431 · Learned box-quality scorer cho B1 (một forward) · Kaggle T4

**Câu hỏi:** đặc trưng lấy từ P-map của **cùng một forward B1** (phân bố xác suất trong box, vòng quanh box, hình dạng) có giúp xếp hạng box tốt hơn raw score không? Phải hơn cả **ngưỡng raw được chọn lại trên cùng tập**, tức không chỉ là calibration.

- **Dữ liệu:** chỉ TRAIN. Gồm 1.600 nguồn fit, 400 nguồn calibration, 10 nguồn smoke; mỗi nguồn có Clean + LR2. **Không** dùng A/B để fit, **không** mở TEST.
- **Model:** logistic L2 cố định, không quét. Đối chứng là RAW (ngưỡng raw chọn trên calibration) và SCORE_ONLY (kiểm tính hợp lệ).
- **Spec khóa trước:** `docs/research/janek/boxquality_spec_20261124_janek.md`. Calibration chỉ là sàng lọc, **không** phải xác nhận; chưa được gọi là cải tiến.

### Trước khi chạy
1. Kaggle → **New Dataset**, tải `cs431-boxquality-janek-fixed.zip` lên (Private).
2. Import notebook này. Bật **GPU T4 x2** (hoặc T4), **Internet On**, **Add Input** chọn dataset vừa tạo.
3. Chạy **Ô 1 → Ô 4**, đọc kết luận ở Ô 4, rồi mới chạy **Ô 5 → Ô 7**.

**Thời gian: chưa đo.** Ô 2: 5–40 phút. Ô 4 in số **đo thật** cho 4.000 task. Ảnh TRAIN (549 MiB) được tải từng file từ zip TextVQA chính thức; không tải cả 7 GB.

**Dừng sau Ô 4 nếu:** một kiểm nhất quán sai, hoặc dự kiến > 3 giờ. Gửi Claude output Ô 4, không tự đổi cấu hình.

**Trạng thái:** notebook đã tạo và kiểm cú pháp ở local; các bước CPU đã chạy thử trên dữ liệu giả; **chưa chạy GPU**.

## Ô 1 · Cấu hình, tìm dataset, kiểm GPU

Kiểm SHA của mọi file trong dataset. Ghim **GPU 0** (`CUDA_VISIBLE_DEVICES=0`), vì code của Tuấn yêu cầu đúng một GPU.

In [ ]:
# ===================== Ô 1 · CẤU HÌNH =====================
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

PIN = dict(
    MONKEY_V2_URL='https://github.com/ncat218/MONKEY_V2.git',
    MONKEY_V2_COMMIT='dc19b85c8f73d56dd0510165069597363fe25380',
    MONKEYOCRV2_URL='https://github.com/Yuliang-Liu/MonkeyOCRv2.git',
    MONKEYOCRV2_COMMIT='da96afede92f198c19aca74e1e5f5aa7f4c26af2',
    TORCH='2.9.0', TORCHVISION='0.24.0', TORCHAUDIO='2.9.0', TORCH_INDEX='https://download.pytorch.org/whl/cu128',
    MMENGINE='0.10.7', MMCV='2.0.1', MMOCR='1.0.1', PILLOW='11.3.0', OPENCV='4.10.0.84',
    DB_PP_SHA256='9bfff8aaaca103dd08d7565f2b7a5410ab20a6427fe1f0baad5a5d04d207c8b8',
    BUNDLE_MANIFEST_SHA256='e06ef54e720b43bcb658fd734291fb1b6e4a08db13a8bd4048f16cfd3a2dc573',
)
W = Path('/kaggle/working'); OUT = W/'cs431_out'; LOGS = OUT/'logs'
T = Path('/tmp/cs431')
for p in (OUT, LOGS, T):
    p.mkdir(parents=True, exist_ok=True)

def fail(msg):
    raise SystemExit('LỖI: ' + msg)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

found = [p for p in Path('/kaggle/input').glob('**/KAGGLE_MANIFEST.json') if json.loads(p.read_text()).get('kind') == 'boxquality']
if not found:
    zips = [z for z in Path('/kaggle/input').glob('**/*.zip') if 'boxquality' in z.name.lower()]
    if not zips:
        fail('Không thấy dataset cs431-boxquality. Bấm "Add Input" và chọn dataset đó.')
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(T/'dataset')
    found = [p for p in (T/'dataset').glob('**/KAGGLE_MANIFEST.json') if json.loads(p.read_text()).get('kind') == 'boxquality']
if len(found) != 1:
    fail(f'Thấy {len(found)} dataset boxquality; chỉ gắn một.')
DS = found[0].parent
manifest = json.loads(found[0].read_text())
bad = []
for rel, info in manifest['files'].items():
    path = DS/rel
    if not path.is_file():
        bad.append(dict(file=rel, reason='MISSING', expected_sha256=info['sha256']))
    else:
        actual = sha256(path)
        if actual != info['sha256']:
            bad.append(dict(file=rel, reason='SHA_MISMATCH', expected_sha256=info['sha256'], actual_sha256=actual, bytes=path.stat().st_size))
if bad:
    print(json.dumps(bad, indent=2))
    fail('Dataset thiếu hoặc thay đổi file; xem chi tiết phía trên.')
if not (DS/'code/mixab/boxquality_janek.py').is_file() or not (DS/'inputs/train_rows.gzip.bin').is_file():
    fail('Hãy gắn dataset cs431-boxquality-janek-fixed, bỏ dataset cũ.')
print(f'Dataset: {DS} ({len(manifest["files"])} file, SHA khớp) · selection {manifest["selection_sha256"][:12]}…')

CODE = T/'code'
if CODE.exists():
    shutil.rmtree(CODE)
shutil.copytree(DS/'code', CODE)
code_sha = hashlib.sha256((DS/'code/CODE_MANIFEST.json').read_bytes()).hexdigest()
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
import importlib
import mixab.colab_helpers_janek as ch
importlib.reload(ch)

gpus = (ch.output_of(['nvidia-smi', '--query-gpu=index,name,driver_version,memory.total', '--format=csv,noheader']) or '').splitlines()
if not gpus:
    fail('Không thấy GPU. Settings -> Accelerator -> GPU T4 x2 (hoặc GPU T4).')
print('GPU của máy:'); [print('  ', g) for g in gpus]
GPU = ch.gpu_label(gpus[0].split(',')[1].strip())
CONFIG_NOTE = f'{len(gpus)} GPU trên máy; chỉ dùng GPU 0 qua CUDA_VISIBLE_DEVICES=0'
print(CONFIG_NOTE)

PY = sys.executable
MV2, MOCR = T/'MONKEY_V2', T/'MonkeyOCRv2'
BUNDLE = MV2/'docs/handoff/artifacts/pr_quad_audit_v1'
WEIGHTS, CLEAN, DUMP, A2R = DS/'weights', DS/'clean', DS/'dump', DS/'a2r/B1_source_counts.npz'
ENCODER = T/'encoder'
if not (ENCODER/'model.safetensors').is_file():
    shutil.copytree(DS/'encoder', ENCODER, dirs_exist_ok=True)
PREPARED, CONFIG_PATH = T/'prepared', T/'inference_config_kaggle.py'
ENV = dict(MONKEY_V2_ROOT=str(MV2), PYTHONPATH=os.pathsep.join((str(CODE), str(MV2))), CUDA_VISIBLE_DEVICES='0')
print(f'Python {sys.version.split()[0]} · code manifest {code_sha[:12]}… · kết quả: {OUT}')

## Ô 2 · Cài môi trường (cùng phiên bản với lần chạy dump T4 trên Colab)

Thứ tự: torch 2.9.0 cu128 → mmengine 0.10.7 → **MMCV 2.0.1** (dùng wheel trong dataset nếu cùng phiên bản Python, nếu không thì **build từ source**) → `install.sh` của MonkeyOCRv2 (MMOCR 1.0.1 + add-on) → ghim Pillow 11.3.0 và OpenCV 4.10.0.84.
- Build MMCV có thể mất **20–40 phút** (ước lượng, chưa đo trên Kaggle). Wheel build xong được chép vào `/kaggle/working/cs431_out/wheels/`: **tải về và thêm vào dataset** để lần sau khỏi build.
- Cần `nvcc` CUDA **12.x** để build. Ô này dùng toolkit có sẵn trên Kaggle (ưu tiên 12.8); nếu không có thì cài `cuda-toolkit-12-8` (không cài driver).
- Kiểm cuối chạy trong tiến trình mới: phiên bản, `MonkeyOCRv2ViTAEBackbone` đã đăng ký, SHA của DBPostprocessor phải đúng bản Tuấn ghim.
- Nếu Kaggle báo cần *Restart*, restart rồi chạy lại **Ô 1** và **Ô 2** (phần đã cài sẽ bỏ qua).

In [ ]:
# ===================== Ô 2 · CÀI MÔI TRƯỜNG =====================
import platform, re
LOG = LOGS/f'install_{GPU}.log'
pyver = sys.version_info
if not ((3, 10) <= pyver[:2] <= (3, 13)):
    print(f'CẢNH BÁO: Python {platform.python_version()} chưa được kiểm với MMCV 2.0.1.')

def ver(name):
    return ch.package_version(name)

def pip(*args):
    ch.run([PY, '-m', 'pip', 'install', '-q', *args], LOG, title='pip install ' + ' '.join(args))

if ver('torch') != f"{PIN['TORCH']}+cu128" or ver('torchvision') != f"{PIN['TORCHVISION']}+cu128":
    pip(f"torch=={PIN['TORCH']}", f"torchvision=={PIN['TORCHVISION']}", '--index-url', PIN['TORCH_INDEX'])
if ver('torchaudio') != f"{PIN['TORCHAUDIO']}+cu128":
    pip('--no-deps', f"torchaudio=={PIN['TORCHAUDIO']}+cu128", '--index-url', PIN['TORCH_INDEX'])
cuda_ok = ch.output_of([PY, '-c', 'import torch;print(torch.version.cuda, torch.cuda.is_available())'], env=ENV)
print('torch CUDA:', cuda_ok)
if not cuda_ok or 'True' not in cuda_ok:
    fail('torch không dùng được CUDA (driver Kaggle quá cũ cho cu128?). Gửi log: ' + str(LOG))
if ver('mmengine') != PIN['MMENGINE']:
    pip(f"mmengine=={PIN['MMENGINE']}")
pip('setuptools<81', 'wheel', 'ninja')

def mmcv_ok():
    if ver('mmcv') != PIN['MMCV']:
        return False
    probe = subprocess.run([PY, '-c', 'from mmcv.ops import nms; print("IMPORT_NMS_OK")'], capture_output=True, text=True)
    with LOG.open('a', encoding='utf-8') as log:
        log.write('\nMMCV import probe:\n' + probe.stdout + probe.stderr)
    return probe.returncode == 0 and 'IMPORT_NMS_OK' in probe.stdout.splitlines()

def find_toolkit():
    """CUDA 12.x toolkit with nvcc + cuda.h; prefer 12.8. torch only refuses a different MAJOR version."""
    import glob
    homes = ['/usr/local/cuda-12.8', os.environ.get('CUDA_HOME'), '/usr/local/cuda', *sorted(glob.glob('/usr/local/cuda-12*'))]
    nvcc = shutil.which('nvcc')
    if nvcc:
        homes.append(str(Path(nvcc).resolve().parents[1]))
    best = None
    for home in homes:
        if not home or not (Path(home)/'bin/nvcc').is_file() or not (Path(home)/'include/cuda.h').is_file():
            continue
        m = re.search(r'release (\d+)\.(\d+)', ch.output_of([str(Path(home)/'bin/nvcc'), '--version']) or '')
        if m and m.group(1) == '12':
            cand = (int(m.group(2)) == 8, int(m.group(2)), str(Path(home).resolve()))
            best = max(best, cand) if best else cand
    return best

tag = f'cp{pyver[0]}{pyver[1]}'
if not mmcv_ok():
    wheels = sorted((DS/'wheels').glob(f"mmcv-{PIN['MMCV']}-{tag}-*.whl")) + sorted(OUT.glob(f"wheels/mmcv-{PIN['MMCV']}-{tag}-*.whl"))
    if wheels:
        print('Cài MMCV từ wheel có sẵn:', wheels[0].name)
        pip('--no-deps', str(wheels[0]))
    else:
        tk = find_toolkit()
        if not tk:
            print('Không có CUDA toolkit 12.x. Cài cuda-toolkit-12-8 từ NVIDIA (không cài driver)...')
            distro = platform.freedesktop_os_release()
            repo = 'ubuntu' + distro.get('VERSION_ID', '22.04').replace('.', '')
            from urllib.request import urlretrieve
            keyring = T/'cuda-keyring_1.1-1_all.deb'
            urlretrieve(f'https://developer.download.nvidia.com/compute/cuda/repos/{repo}/x86_64/cuda-keyring_1.1-1_all.deb', keyring)
            ch.run(['dpkg', '-i', keyring], LOG)
            ch.run(['apt-get', 'update'], LOG)
            ch.run(['apt-get', 'install', '-y', '--no-install-recommends', 'cuda-toolkit-12-8'], LOG, env={'DEBIAN_FRONTEND': 'noninteractive'})
            tk = find_toolkit()
            if not tk:
                fail('Đã cài nhưng vẫn không thấy nvcc 12.x; xem ' + str(LOG))
        cuda_home = tk[2]
        print(f'Dùng CUDA toolkit {cuda_home} (12.{tk[1]})' + ('' if tk[0] else ' — khác 12.8, torch chỉ cảnh báo khác minor version'))
        src = T/'mmcv_src'
        if not (src/'setup.py').is_file():
            ch.run(['git', 'clone', '-b', f"v{PIN['MMCV']}", '--depth', '1', 'https://github.com/open-mmlab/mmcv.git', src], LOG)
        setup = (src/'setup.py').read_text()
        setup = setup.replace('-std=c++14', '-std=c++17')
        old_v = "        exec(compile(f.read(), version_file, 'exec'))\n    return locals()['__version__']"
        new_v = "        namespace = {}\n        exec(compile(f.read(), version_file, 'exec'), namespace)\n    return namespace['__version__']"
        if old_v in setup:
            setup = setup.replace(old_v, new_v)
        elif new_v not in setup:
            fail('MMCV setup.py khác cấu trúc đã kiểm.')
        (src/'setup.py').write_text(setup)
        print(f'Build MMCV ({os.cpu_count()} CPU). Có thể mất lâu...')
        build_env = dict(CUDA_HOME=cuda_home, MMCV_WITH_OPS='1', MAX_JOBS=str(os.cpu_count() or 2),
                         PATH=f"{cuda_home}/bin:{os.environ.get('PATH', '')}", TORCH_CUDA_ARCH_LIST='7.5;8.0')
        wheel_tmp = T/'wheelhouse'
        ch.run([PY, '-m', 'pip', 'wheel', '-v', '--no-build-isolation', '--no-deps', '-w', wheel_tmp, src], LOG,
               env=build_env, title='build MMCV wheel')
        built = sorted(wheel_tmp.glob('mmcv-*.whl'))
        if not built:
            fail('Build MMCV không tạo ra wheel; xem ' + str(LOG))
        (OUT/'wheels').mkdir(exist_ok=True)
        shutil.copy2(built[-1], OUT/'wheels'/built[-1].name)
        print('Đã lưu wheel để tải về:', OUT/'wheels'/built[-1].name)
        pip('--no-deps', str(built[-1]))
    if not mmcv_ok():
        fail('MMCV chưa import được mmcv.ops.nms; xem ' + str(LOG))

if not (MOCR/'.git').is_dir():
    ch.run(['git', 'clone', PIN['MONKEYOCRV2_URL'], MOCR], LOG)
ch.run(['git', '-C', MOCR, 'checkout', '--detach', PIN['MONKEYOCRV2_COMMIT']], LOG)
addon = MOCR/'detection/mmocr'
mmocr_root = addon/'mmocr'
backbone_file = mmocr_root/'mmocr/models/common/backbones/monkeyocr_v2_vitae.py'
mmocr_loc = ch.output_of([PY, '-c', 'import mmocr,os;print(os.path.dirname(mmocr.__file__))'])
if not (ver('mmocr') == PIN['MMOCR'] and backbone_file.is_file() and mmocr_loc and str(mmocr_root) in mmocr_loc):
    if not (mmocr_root/'.git').is_dir():
        ch.run(['git', 'clone', '--branch', 'v1.0.1', '--depth', '1', 'https://github.com/open-mmlab/mmocr.git', mmocr_root], LOG)
    setup_path = mmocr_root/'setup.py'
    setup = setup_path.read_text()
    old_exec = "        exec(compile(f.read(), version_file, 'exec'))"
    new_exec = "        namespace = {}\n        exec(compile(f.read(), version_file, 'exec'), namespace)"
    if old_exec in setup:
        setup = setup.replace(old_exec, new_exec)
        setup = setup.replace("locals()['short_version']", "namespace['short_version']").replace("locals()['__version__']", "namespace['__version__']")
        setup_path.write_text(setup)
    elif new_exec not in setup:
        fail('MMOCR setup.py khác cấu trúc đã kiểm.')
    ch.run(['bash', 'install.sh'], LOG, cwd=addon, title='MonkeyOCRv2 detection/mmocr/install.sh',
           env=dict(PATH=f"{Path(PY).parent}:{os.environ.get('PATH', '')}", PIP_VERBOSE='1'))
if ver('Pillow') != PIN['PILLOW']:
    pip(f"Pillow=={PIN['PILLOW']}")
if ver('opencv-python') != PIN['OPENCV'] or any(ver(n) for n in ('opencv-python-headless', 'opencv-contrib-python')):
    ch.run([PY, '-m', 'pip', 'uninstall', '-y', 'opencv-python-headless', 'opencv-contrib-python', 'opencv-contrib-python-headless'], LOG)
    pip('--force-reinstall', '--no-deps', f"opencv-python=={PIN['OPENCV']}")
if ver('shapely') is None:
    pip('shapely')

VERIFY = """
import hashlib, importlib.metadata as m, importlib.util, json, torch
from mmocr.utils import register_all_modules
from mmocr.registry import MODELS
from mmcv.ops import nms
register_all_modules(init_default_scope=True)
spec = importlib.util.find_spec('mmocr.models.textdet.postprocessors.db_postprocessor')
out = dict(versions={k: m.version(k) for k in ('torch','torchvision','mmengine','mmcv','mmdet','mmocr','Pillow','opencv-python','numpy','shapely')},
           cuda=torch.cuda.is_available(), device_count_visible=torch.cuda.device_count(), device=torch.cuda.get_device_name(0),
           backbone_registered=MODELS.get('MonkeyOCRv2ViTAEBackbone') is not None,
           db_postprocessor_sha256=hashlib.sha256(open(spec.origin,'rb').read()).hexdigest())
print(json.dumps(out))
"""
raw = ch.output_of([PY, '-c', VERIFY], env=ENV)
if not raw:
    fail('Kiểm cuối môi trường lỗi khi import; xem ' + str(LOG))
env_report = json.loads(raw.splitlines()[-1])
(OUT/f'env_{GPU}.json').write_text(json.dumps(dict(env_report, pins=PIN, gpus=gpus, config=CONFIG_NOTE), indent=1))
print(json.dumps(env_report, indent=1))
problems = []
if env_report['db_postprocessor_sha256'] != PIN['DB_PP_SHA256']:
    problems.append('DBPostprocessor khác bản Tuấn ghim')
if not env_report['backbone_registered']:
    problems.append('MonkeyOCRv2ViTAEBackbone chưa đăng ký')
if env_report['versions']['Pillow'] != PIN['PILLOW']:
    problems.append('Pillow không phải 11.3.0 (cần để tạo LR2 đúng byte)')
if env_report['device_count_visible'] != 1:
    problems.append('Tiến trình đo thấy nhiều hơn 1 GPU')
if problems:
    fail('Môi trường chưa đạt: ' + '; '.join(problems))
print('\nÔ 2 xong.')

## Ô 3 · Repo của Tuấn, config, B1, kiểm lại nhãn TRAIN từ nguồn gốc, ảnh smoke

- Clone `MONKEY_V2` @ `dc19b85`, vá config (chỉ đường dẫn encoder), kiểm SHA của B1.
- Tải TextOCR train JSON chính thức (280 MB), kiểm SHA `5bac8303…`. **Dựng lại nhãn TRAIN** và so với `inputs/train_rows.jsonl.gz` (phải trùng từng dòng).
- Tải 10 ảnh smoke từ zip TextVQA chính thức (CRC + SHA với `train_ids.csv`), tạo LR2 bằng `native_lr2` của Tuấn, chạy test CPU.

In [ ]:
# ===================== Ô 3 · DỮ LIỆU =====================
import csv, gzip
from pathlib import PureWindowsPath
import mixab.boxquality_janek as bq
importlib.reload(bq)
LOG = LOGS/f'data_{GPU}.log'
if not (MV2/'.git').is_dir():
    ch.run(['git', 'clone', PIN['MONKEY_V2_URL'], MV2], LOG, env=dict(GIT_LFS_SKIP_SMUDGE='1'))
ch.run(['git', '-C', MV2, 'checkout', '--detach', PIN['MONKEY_V2_COMMIT']], LOG, env=dict(GIT_LFS_SKIP_SMUDGE='1'))
if ch.output_of(['git', '-C', MV2, 'status', '--porcelain', '--untracked-files=no']):
    fail('Checkout MONKEY_V2 bị sửa.')
if sha256(BUNDLE/'bundle_manifest.json') != PIN['BUNDLE_MANIFEST_SHA256']:
    fail('bundle_manifest.json khác bản đã khóa.')
CONFIG_PATH.write_text(ch.patch_inference_config((MV2/'docs/handoff/inference_config.py').read_text(encoding='utf-8'), ENCODER), encoding='utf-8')
m = json.loads((WEIGHTS/'merge_manifest.json').read_text())
b1 = next(x for x in m['models'] if x['alpha'] == '1')
if sha256(WEIGHTS/PureWindowsPath(b1['checkpoint']).name) != b1['checkpoint_sha256']:
    fail('alpha_1p00.pth (B1) sai SHA.')
INP = T/'boxquality_inputs_verified'
shutil.copytree(DS/'inputs', INP, dirs_exist_ok=True)
shutil.copyfile(INP/'train_rows.gzip.bin', INP/'train_rows.jsonl.gz')
if sha256(INP/'train_rows.jsonl.gz') != manifest['train_rows_sha256']:
    fail('Nhãn TRAIN sau khôi phục tên không khớp SHA gốc.')
if sha256(INP/'train_ids.csv') != bq.TRAIN_IDS_SHA256:
    fail('train_ids.csv sai SHA.')
TXT = T/'textocr/TextOCR_0.1_train.json'
TXT.parent.mkdir(parents=True, exist_ok=True)
if not (TXT.is_file() and TXT.stat().st_size == bq.TEXTOCR_BYTES and sha256(TXT) == bq.TEXTOCR_TRAIN_SHA256):
    ch.run(['wget', '-c', '-q', '-O', TXT, bq.TEXTOCR_URL], LOG, title='download TextOCR train JSON (280 MB)')
if sha256(TXT) != bq.TEXTOCR_TRAIN_SHA256:
    fail('TextOCR train JSON sai SHA; chạy lại ô để tải tiếp.')
RC = T/'rows_check'
ch.run([PY, '-m', 'mixab.boxquality_janek', 'rows', '--textocr', TXT, '--train-ids', INP/'train_ids.csv', '--output', RC], LOG,
       cwd=CODE, env=ENV, title='rebuild TRAIN label rows')
a_ = [json.loads(l) for l in gzip.open(RC/'train_rows.jsonl.gz', 'rt', encoding='utf-8')]
b_ = [json.loads(l) for l in gzip.open(INP/'train_rows.jsonl.gz', 'rt', encoding='utf-8')]
if a_ != b_:
    fail('Nhãn TRAIN dựng lại khác dataset.')
print(f'Nhãn TRAIN dựng lại trùng dataset: {len(a_)} nguồn', json.loads((RC/'rows_receipt.json').read_text())['roles'])
TRAIN_CLEAN = T/'train_clean'
SEL = list(csv.DictReader(open(INP/'selection.csv', encoding='utf-8')))
tic = time.time()
r = bq.fetch_members([(x['clean_relpath'], x['clean_sha256']) for x in SEL if x['role'] == 'smoke'], TRAIN_CLEAN)
print(f"Ảnh smoke: {r} ({time.time() - tic:.1f} s)")
ch.run([PY, '-m', 'mixab.boxquality_janek', 'prepare', '--rows', INP/'train_rows.jsonl.gz', '--clean-root', TRAIN_CLEAN,
        '--prepared-root', PREPARED, '--roles', 'smoke'], LOG, cwd=CODE, env=ENV, title='prepare smoke LR2')
ch.run([PY, '-m', 'unittest', 'discover', '-s', 'tests', '-t', '.'], LOG, cwd=CODE, env=ENV, title='mixab unit tests')
print('\nÔ 3 xong.')

## Ô 4 · Smoke 10 nguồn TRAIN (20 task) + kiểm nhất quán + chi phí thật

1. **Trích đặc trưng:** đường đặc trưng phải cho polygon và raw **trùng tuyệt đối** `capture_candidates` của Tuấn ở cả 20 task.
2. **`smokecheck` (CPU):**
   - evaluator chính thức khớp `CandidateMatcher` ở B1 `.40/.60`;
   - raw feature bằng raw score;
   - đo thời gian gán nhãn.
3. **`devcheck`:** chạy cùng đường trên 5 ảnh dev LR2 và so với dump T4 cũ. Phải trùng tuyệt đối. Bước này chỉ kiểm pipeline, **không** đọc nhãn dev.
4. In chi phí: forward, post-process thường, post-process + đặc trưng (median / p95), VRAM đỉnh, dự kiến cho 4.000 task.

In [ ]:
# ===================== Ô 4 · SMOKE =====================
SM = OUT/'smoke_extract'
if not (SM/'extract_summary.json').is_file():
    ch.run([PY, '-m', 'mixab.boxquality_janek', 'extract', '--rows', INP/'train_rows.jsonl.gz', '--clean-root', TRAIN_CLEAN,
            '--prepared-root', PREPARED, '--roles', 'smoke', '--model-config', CONFIG_PATH, '--weights-dir', WEIGHTS,
            '--consistency', '--shard', '20', '--output', SM], LOGS/'smoke_extract.log', cwd=CODE, env=ENV, title='smoke extract')
ch.run([PY, '-m', 'mixab.boxquality_janek', 'smokecheck', '--bundle', BUNDLE, '--rows', INP/'train_rows.jsonl.gz', '--extract-dir', SM,
        '--output', OUT/'smokecheck'], LOGS/'smokecheck.log', cwd=CODE, env=ENV, title='smokecheck')
if not (OUT/'devcheck/devcheck.json').is_file():
    ch.run([PY, '-m', 'mixab.boxquality_janek', 'devcheck', '--bundle', BUNDLE, '--sources', INP/'devcheck_sources.json',
            '--clean-root', DS/'dev_clean', '--prepared-root', T/'dev_prepared', '--model-config', CONFIG_PATH,
            '--weights-dir', WEIGHTS, '--dump', DUMP, '--output', OUT/'devcheck'], LOGS/'devcheck.log', cwd=CODE, env=ENV, title='devcheck')
s = json.loads((SM/'extract_summary.json').read_text())
k = json.loads((OUT/'smokecheck/smokecheck.json').read_text())
d = json.loads((OUT/'devcheck/devcheck.json').read_text())
print(f"Trích: {s['tasks']} task, {s['candidates']} ứng viên · nhất quán với capture_candidates: {s['consistency_checked']} task kiểm, tất cả đúng = {s['consistency_all_true']}")
print(f"ms: {s['ms']} · tỉ số post-process có đặc trưng/thường (median) {s['feature_overhead_ratio_median']:.2f} · VRAM đỉnh {s['peak_gpu_mib']:.0f} MiB")
print(f"smokecheck: evaluator chính thức khớp = {k['official_evaluator_agrees']}, raw feature đúng = {k['raw_feature_ok']}, gán nhãn {k['label_seconds_per_task_median']*1000:.1f} ms/task")
print(f"devcheck: {d['identical']}/{d['sources']} ảnh dev LR2 trùng tuyệt đối dump T4 ·", [(x['source'][:8], x['dump_candidates'], x['live_candidates'], x['max_abs_raw_diff']) for x in d['detail']])
proj = s['projected_minutes_for_4000_tasks']
print(f"\nDự kiến trích 4.000 task: {proj:.0f} phút (đo thật trên smoke, gồm kiểm SHA + post-process 2 lần ở task kiểm)")
ok = s['consistency_all_true'] and k['official_evaluator_agrees'] and k['raw_feature_ok'] and d['identical'] == d['sources']
if not ok:
    print('DỪNG: có kiểm nhất quán sai. Gửi Claude output ô này.')
elif proj > 180:
    print('DỪNG: dự kiến vượt 3 giờ. Báo Claude.')
else:
    print('Được chạy Ô 5.')

## Ô 5 · Tải 2.000 ảnh TRAIN, dựng LR2, trích 4.000 task (GPU)

- Ảnh được tải từng file (CRC + SHA-256). Ảnh đã có và đúng SHA sẽ bỏ qua.
- Trích lưu sau mỗi 50 task vào `cs431_out/extract/`. Bị ngắt **trong cùng phiên** thì chạy lại ô này, phần xong sẽ bỏ qua.
- Cứ 50 task lại kiểm nhất quán với `capture_candidates` một lần.
- Nếu phiên sắp hết giờ: chạy Ô 7 để tải phần đã xong.

In [ ]:
# ===================== Ô 5 · TRÍCH TRAIN =====================
tic = time.time()
r = bq.fetch_members([(x['clean_relpath'], x['clean_sha256']) for x in SEL if x['role'] in ('fit', 'calibration')], TRAIN_CLEAN)
print(f"Ảnh fit + calibration: {r} ({(time.time() - tic)/60:.1f} phút)")
ch.run([PY, '-m', 'mixab.boxquality_janek', 'prepare', '--rows', INP/'train_rows.jsonl.gz', '--clean-root', TRAIN_CLEAN,
        '--prepared-root', PREPARED, '--roles', 'fit,calibration'], LOGS/'prepare_train.log', cwd=CODE, env=ENV, title='prepare TRAIN LR2')
EX = OUT/'extract'
ch.run([PY, '-m', 'mixab.boxquality_janek', 'extract', '--rows', INP/'train_rows.jsonl.gz', '--clean-root', TRAIN_CLEAN,
        '--prepared-root', PREPARED, '--roles', 'fit,calibration', '--model-config', CONFIG_PATH, '--weights-dir', WEIGHTS,
        '--shard', '50', '--output', EX], LOGS/f'extract_{time.strftime("%Y%m%d-%H%M%S")}.log', cwd=CODE, env=ENV, title='TRAIN extract')
s = json.loads((EX/'extract_summary.json').read_text())
print({k_: s[k_] for k_ in ('tasks', 'candidates', 'consistency_checked', 'consistency_all_true', 'ms', 'peak_gpu_mib')})

## Ô 6 · Fit + chọn ngưỡng trên TRAIN-calibration (CPU)

- Gán nhãn theo evaluator. Fit FULL và SCORE_ONLY trên 1.600 nguồn fit.
- Chọn ngưỡng cho RAW, SCORE_ONLY và FULL trên **cùng** 400 nguồn calibration (luật khóa ở spec mục 6).
- Báo FULL − RAW, FULL − F, RAW − F kèm CI ghép cặp theo nguồn, và nhãn khóa.
- Xuất model, ngưỡng, đường cong, và bảng review 32 ảnh (24 FP điểm cao không chồng GT + 8 TP) để xem chữ thiếu nhãn.

In [ ]:
# ===================== Ô 6 · FIT =====================
FT = OUT/'fit'
if not (FT/'boxquality_fit.json').is_file():
    ch.run([PY, '-m', 'mixab.boxquality_janek', 'fit', '--bundle', BUNDLE, '--rows', INP/'train_rows.jsonl.gz', '--extract-dir', OUT/'extract',
            '--output', FT, '--clean-root', TRAIN_CLEAN], LOGS/'fit.log', cwd=CODE, env=ENV, title='fit + calibrate')
r = json.loads((FT/'boxquality_fit.json').read_text())
print('Ứng viên:', r['candidates'])
print('F .40/.60 (calibration):', {c: round(r['F_040_060_calibration'][c]['hmean']*100, 2) for c in ('clean', 'lr2')})
for name, v in r['chosen'].items():
    print(f"  {name:10} cut {v['cut']:.4f} · Clean {v['clean']['hmean']*100:.2f} · LR2 {v['lr2']['hmean']*100:.2f} · mean {v['mean_hmean']*100:.2f} · tiny LR2 {v['lr2']['tiny_recall']*100:.1f}")
for k_ in ('full_minus_raw', 'score_only_minus_raw', 'full_minus_F', 'raw_minus_F'):
    print(f"  {k_:22} {r[k_]['delta_pp']:+.2f} pp [{r[k_]['ci95_pp'][0]:+.2f}; {r[k_]['ci95_pp'][1]:+.2f}]")
print('\nNhãn:', r['outcome'], '(chỉ là sàng lọc trên TRAIN-calibration; không phải xác nhận, không phải cải tiến)')

## Ô 7 · Đóng gói

Tạo `/kaggle/working/cs431_boxquality_<GPU>_<thời gian>.zip`, gồm toàn bộ `cs431_out` trừ wheel. Ảnh TRAIN nằm ở `/tmp` nên không có trong zip. Tải ở khung **Output** và gửi Claude (hoặc báo Claude tải bằng Kaggle API).

In [ ]:
# ===================== Ô 7 · ĐÓNG GÓI =====================
pack = W/f'cs431_boxquality_{GPU}_{time.strftime("%Y%m%d-%H%M%S")}.zip'
with zipfile.ZipFile(pack, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and p.suffix != '.whl':
            zf.write(p, p.relative_to(OUT).as_posix())
    zf.writestr('code_manifest_sha256.txt', code_sha + '\n')
print('Đã đóng gói:', pack, f'({pack.stat().st_size/2**20:.1f} MiB)')